# 10 Results Summary

This notebook is the final summary notebook for `term_project`. It collects results from notebooks `00` through `10` where result artifacts are available.

Covered experiments:

- `00_cache_data.ipynb`: cached quadtree vectors and GT lookup files.
- `01_baseline.ipynb`: original HNSW WeightedJaccard baseline.
- `02_mlp_cosine.ipynb`: MLP + cosine index, with/without WJ rerank.
- `03_neural_minhash.ipynb`: Neural MinHash, plus original-WJ rerank ablation.
- `04_two_stage_mlp_wj_rerank.ipynb`: two-stage MLP candidate generation + exact WJ rerank.
- `05_mlp_wj_distillation.ipynb`: random-negative WJ distillation ablation.
- `06_mlp_hard_negative_distillation.ipynb`: hard-negative WJ distillation.
- `07_adaptive_k_rerank.ipynb`: adaptive-K reranking.
- `08_listwise_wj_distillation.ipynb`: listwise WJ distillation ablation.
- `09_raw_polygon_pointnet_wj_distill.ipynb`: raw-geometry PointNet ablation.

Note: the old `04_results.ipynb` was moved to the end and renamed to this notebook so that results are last in the sequence.


In [1]:
import math
import pickle
from pathlib import Path

RESULT_FILES = {
    "mlp_cosine": Path("/tmp/results_mlp_cosine.pkl"),
    "minhash": Path("/tmp/results_minhash.pkl"),
    "two_stage_mlp_wj": Path("/tmp/results_two_stage_mlp_wj.pkl"),
    "hardneg_wjdistill": Path("/tmp/results_hardneg_wjdistill.pkl"),
    "adaptive_k": Path("/tmp/results_adaptive_k.pkl"),
    "listwise_wjdistill": Path("/tmp/results_listwise_wjdistill.pkl"),
    "minhash_rerank": Path("/tmp/results_minhash_rerank.pkl"),
    "raw_pointnet": Path("/tmp/results_raw_pointnet_wjdistill.pkl"),
}

def load_pickle(path):
    if not path.exists():
        return None
    with open(path, "rb") as f:
        return pickle.load(f)

results = {name: load_pickle(path) for name, path in RESULT_FILES.items()}
for name, obj in results.items():
    print(f"{name:<22} {'OK' if obj is not None else 'MISSING'}  {RESULT_FILES[name]}")

BASELINE = {
    "10k": {
        10: 0.9966, 50: 0.9986, 100: 0.9990, 500: 0.9974,
        "qps": 246.4, "build_s": 51.7, "vec_mb": 564.5, "idx_mb": 20.3,
        "hardware": "CPU 32T",
    },
    "full": {
        10: 0.9925, 50: 0.9953, 100: 0.9963, 500: 0.9864,
        "qps": 610.8, "build_s": 559.7, "vec_mb": 12998.5, "idx_mb": 11519.4,
        "hardware": "CPU 32T",
    },
}

RANDOM_DISTILL_10K = {
    "k500_wj_rerank": {10: 0.9912, 50: 0.9770, 100: 0.9518, 500: 0.8553, "qps": 3419.2},
    "k1000_wj_rerank": {10: 0.9949, 50: 0.9869, 100: 0.9686, 500: 0.8857, "qps": 1850.0},
    "quality": {"base_gt": 0.9923, "base_rand": 0.7966, "base_gap": 0.1957,
                "distill_gt": 0.9883, "distill_rand": 0.6712, "distill_gap": 0.3171},
}

print("\nHard-coded baseline and random-negative distillation ablation loaded.")


mlp_cosine             OK  /tmp/results_mlp_cosine.pkl
minhash                OK  /tmp/results_minhash.pkl
two_stage_mlp_wj       OK  /tmp/results_two_stage_mlp_wj.pkl
hardneg_wjdistill      OK  /tmp/results_hardneg_wjdistill.pkl
adaptive_k             OK  /tmp/results_adaptive_k.pkl
listwise_wjdistill     OK  /tmp/results_listwise_wjdistill.pkl
minhash_rerank         OK  /tmp/results_minhash_rerank.pkl
raw_pointnet           OK  /tmp/results_raw_pointnet_wjdistill.pkl

Hard-coded baseline and random-negative distillation ablation loaded.


In [2]:
def f4(x):
    if x is None:
        return "-"
    try:
        if math.isnan(x):
            return "-"
    except TypeError:
        pass
    return f"{float(x):.4f}"

def f1(x):
    if x is None:
        return "-"
    try:
        if math.isnan(x):
            return "-"
    except TypeError:
        pass
    return f"{float(x):.1f}"

def get(res, k):
    return res.get(k, res.get(str(k))) if isinstance(res, dict) else None

def row(method, dataset, res, hardware="", note=""):
    return {
        "Method": method,
        "Data": dataset,
        "R@10": get(res, 10),
        "R@50": get(res, 50),
        "R@100": get(res, 100),
        "R@500": get(res, 500),
        "QPS": res.get("qps") if isinstance(res, dict) else None,
        "Build(s)": res.get("build_s") if isinstance(res, dict) else None,
        "Vec(MB)": res.get("vec_mb") if isinstance(res, dict) else None,
        "Idx(MB)": res.get("idx_mb") if isinstance(res, dict) else None,
        "HW": hardware,
        "Note": note,
    }

def print_table(rows, title):
    print("\n" + "=" * 150)
    print(title)
    print("=" * 150)
    print(f"{'Method':<42} {'Data':<6} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} {'QPS':>9} {'Build':>9} {'VecMB':>9} {'IdxMB':>9} {'HW':<13} Note")
    print("-" * 150)
    for r in rows:
        print(f"{r['Method']:<42} {r['Data']:<6} {f4(r['R@10']):>7} {f4(r['R@50']):>7} {f4(r['R@100']):>7} {f4(r['R@500']):>7} "
              f"{f1(r['QPS']):>9} {f1(r['Build(s)']):>9} {f1(r['Vec(MB)']):>9} {f1(r['Idx(MB)']):>9} {r['HW']:<13} {r['Note']}")


## Main Results

This table compares the strongest baseline and proposed methods. The baseline is CPU-only HNSW over original quadtree vectors using `WeightedJaccard`. The proposed two-stage methods use CPU HNSW candidate generation and GPU exact original-WJ reranking.


In [3]:
main_rows = []
main_rows.append(row("Baseline HNSW WeightedJaccard", "10k", BASELINE["10k"], "CPU 32T", "original quadtree vectors"))
main_rows.append(row("Baseline HNSW WeightedJaccard", "full", BASELINE["full"], "CPU 32T", "original quadtree vectors"))

# Best hard-negative distilled MLP rows.
hard = results["hardneg_wjdistill"]
if hard:
    runs = hard.get("runs", {})
    if "10k_hardneg_20260427_111822" in runs:
        r = runs["10k_hardneg_20260427_111822"]["hard_eval"]["k500_wj_rerank"]
        main_rows.append(row("Ours HardDist MLP + GPU WJ K=500", "10k", r, "CPU+GPU", "best 10k top-100/top-500 tradeoff"))
    if "full_hardneg_20260427_122346" in runs:
        r = runs["full_hardneg_20260427_122346"]["hard_eval"]["k1000_wj_rerank"]
        main_rows.append(row("Ours HardDist MLP + GPU WJ K=1000", "full", r, "CPU+GPU", "best full speed/top-100 point"))
        r = runs["full_hardneg_20260427_122346"]["hard_eval"]["k2000_wj_rerank"]
        main_rows.append(row("Ours HardDist MLP + GPU WJ K=2000", "full", r, "CPU+GPU", "higher full R@500 point"))

print_table(main_rows, "Main Baseline vs Proposed Results")



Main Baseline vs Proposed Results
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
Baseline HNSW WeightedJaccard              10k     0.9966  0.9986  0.9990  0.9974     246.4      51.7     564.5      20.3 CPU 32T       original quadtree vectors
Baseline HNSW WeightedJaccard              full    0.9925  0.9953  0.9963  0.9864     610.8     559.7   12998.5   11519.4 CPU 32T       original quadtree vectors
Ours HardDist MLP + GPU WJ K=500           10k     0.9966  0.9986  0.9989  0.9678    3752.1       0.2      15.6       2.7 CPU+GPU       best 10k top-100/top-500 tradeoff
Ours HardDist MLP + GPU WJ K=1000          full    0.9926  0.9951  0.9948  0.9034    1000.1     125.1     365.3     401.6 CPU+GPU       best full speed/top-100 point
Ours HardDist

## 02: MLP + Cosine

This section shows that cosine over learned MLP embeddings is fast and compact but is not sufficient without exact WJ reranking.


In [4]:
rows = []
mlp = results["mlp_cosine"]
if mlp:
    for ds in ["10k", "full"]:
        for name, res in mlp.get(ds, {}).items():
            rows.append(row("MLP " + name, ds, res, "CPU 32T", "from 02_mlp_cosine"))
print_table(rows, "02 MLP Cosine Results")



02 MLP Cosine Results
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
MLP cosine_no_rerank                       10k     0.6655  0.8080  0.8464  0.9540   30149.6       0.2      15.6      44.8 CPU 32T       from 02_mlp_cosine
MLP cosine_k100_rerank                     10k     0.9946  0.9581  0.8464       -     405.8       0.2      15.6      44.8 CPU 32T       from 02_mlp_cosine
MLP cosine_k200_rerank                     10k     0.9962  0.9933  0.9720       -     160.6       0.2      15.6      44.8 CPU 32T       from 02_mlp_cosine
MLP bhattacharyya                          10k     0.6256  0.7741  0.8201  0.9461   28225.7       0.2      15.6      19.5 CPU 32T       from 02_mlp_cosine
MLP cosine_no_rerank                       full    0.6581  0.7264

## 03: Neural MinHash

Neural MinHash was tested both as a standalone learned WJ-compatible index and with exact original-WJ reranking. Reranking greatly improves top-k ordering, but candidate inclusion remains weaker than HardDist MLP.


In [5]:
rows = []
mh = results["minhash"]
if mh:
    for ds, res in mh.items():
        rows.append(row("Neural MinHash no rerank", ds, res, "CPU 32T", "learned WJ index"))

mhr = results["minhash_rerank"]
if mhr:
    for run in mhr.get("runs", {}).values():
        for ds, dsres in run.get("results", {}).items():
            for name, res in dsres.items():
                rows.append(row("Neural MinHash + orig WJ " + name.replace("_orig_wj_rerank", ""), ds, res, "CPU+GPU", "candidate WJ over learned vectors"))
print_table(rows, "03 Neural MinHash Results")



03 Neural MinHash Results
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
Neural MinHash no rerank                   10k     0.6586  0.7964  0.8461  0.9571   18604.9       0.3      15.6      16.6 CPU 32T       learned WJ index
Neural MinHash no rerank                   full    0.5821  0.6604  0.6729  0.7068    4503.9      36.6     365.3     631.8 CPU 32T       learned WJ index
Neural MinHash + orig WJ k100              full    0.9701  0.8500  0.6729       -    2009.8      90.5     365.3      76.6 CPU+GPU       candidate WJ over learned vectors
Neural MinHash + orig WJ k200              full    0.9864  0.9440  0.8606       -    1845.2      90.5     365.3      76.6 CPU+GPU       candidate WJ over learned vectors
Neural MinHash + orig WJ k500      

## 04: Two-Stage MLP + Exact WJ Rerank

This is the first two-stage version: MLP cosine candidates followed by exact original WeightedJaccard reranking.


In [6]:
rows = []
ts = results["two_stage_mlp_wj"]
if ts:
    # Support both early flat format and newer runs format.
    if "runs" in ts:
        for run in ts["runs"].values():
            dsroot = run.get("datasets", {})
            for ds, dsres in dsroot.items():
                for name, res in dsres.items():
                    rows.append(row("Two-stage MLP + GPU WJ " + name, ds, res, "CPU+GPU", "from 04"))
    else:
        for ds, dsres in ts.items():
            for name, res in dsres.items():
                rows.append(row("Two-stage MLP + GPU WJ " + name, ds, res, "CPU+GPU", "from 04"))
print_table(rows, "05 Two-Stage MLP + Exact WJ Rerank")



05 Two-Stage MLP + Exact WJ Rerank
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
Two-stage MLP + GPU WJ k500_wj_rerank      10k     0.9966  0.9984  0.9980  0.9540    2903.5       0.2      15.6      32.8 CPU+GPU       from 04
Two-stage MLP + GPU WJ k1000_wj_rerank     10k     0.9966  0.9985  0.9987  0.9825    1766.2       0.2      15.6      32.8 CPU+GPU       from 04
Two-stage MLP + GPU WJ k2000_wj_rerank     10k     0.9966  0.9985  0.9987  0.9825    1763.9       0.2      15.6      32.8 CPU+GPU       from 04


## 05: Random-Negative WJ Distillation Ablation

Random-negative distillation improved GT-vs-random cosine separation but hurt retrieval. This motivated hard-negative distillation.


In [7]:
rows = []
for name, res in RANDOM_DISTILL_10K.items():
    if name == "quality":
        continue
    rows.append(row("RandomNeg WJ Distill " + name, "10k", res, "CPU+GPU", "negative result from 05"))
print_table(rows, "06 Random-Negative Distillation Ablation")
q = RANDOM_DISTILL_10K["quality"]
print("\nEmbedding quality check:")
print(f"Base      GT={q['base_gt']:.4f} | Rand={q['base_rand']:.4f} | Gap={q['base_gap']:.4f}")
print(f"Distilled GT={q['distill_gt']:.4f} | Rand={q['distill_rand']:.4f} | Gap={q['distill_gap']:.4f}")
print("Interpretation: better random separation did not translate to better WJ candidate recall.")



06 Random-Negative Distillation Ablation
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
RandomNeg WJ Distill k500_wj_rerank        10k     0.9912  0.9770  0.9518  0.8553    3419.2         -         -         - CPU+GPU       negative result from 05
RandomNeg WJ Distill k1000_wj_rerank       10k     0.9949  0.9869  0.9686  0.8857    1850.0         -         -         - CPU+GPU       negative result from 05

Embedding quality check:
Base      GT=0.9923 | Rand=0.7966 | Gap=0.1957
Distilled GT=0.9883 | Rand=0.6712 | Gap=0.3171
Interpretation: better random separation did not translate to better WJ candidate recall.


## 06: Hard-Negative WJ Distillation

This is the strongest ML training result. Hard negatives are mined from current cosine candidates that are not WJ ground-truth neighbors.


In [8]:
rows = []
hard = results["hardneg_wjdistill"]
if hard:
    for run_name, run in hard.get("runs", {}).items():
        ds = run.get("config", {}).get("dataset_name", "?")
        for group_name in ["base_eval", "hard_eval"]:
            label = "Base" if group_name == "base_eval" else "HardDist"
            for name, res in run.get(group_name, {}).items():
                rows.append(row(label + " " + name, ds, res, "CPU+GPU", run_name))
print_table(rows, "07 Hard-Negative Distillation Results")



07 Hard-Negative Distillation Results
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
Base k100_wj_rerank                        10k     0.9948  0.9582  0.8465       -    7527.4       0.2      15.6       9.3 CPU+GPU       10k_hardneg_20260427_111822
Base k200_wj_rerank                        10k     0.9964  0.9934  0.9721       -    7099.5       0.2      15.6       9.3 CPU+GPU       10k_hardneg_20260427_111822
Base k500_wj_rerank                        10k     0.9966  0.9984  0.9981  0.9541    3775.9       0.2      15.6       9.3 CPU+GPU       10k_hardneg_20260427_111822
HardDist k100_wj_rerank                    10k     0.9962  0.9724  0.8641       -   10384.4       0.2      15.6       2.7 CPU+GPU       10k_hardneg_20260427_111822
HardDist k200

## 07: Adaptive-K Reranking

Adaptive-K selects a per-query candidate budget. It is useful as a speed/quality knob: lower average K means less exact WJ reranking work.


In [9]:
rows = []
ad = results["adaptive_k"]
if ad:
    for run_name, run in ad.get("runs", {}).items():
        cfg = run.get("config", {})
        ds = cfg.get("dataset_name", "?")
        target = cfg.get("target_r100")
        for k, res in run.get("fixed_results", {}).items():
            r = dict(res)
            r["qps"] = None
            rows.append(row(f"Fixed K={k}", ds, r, "cached", f"{run_name}"))
        ar = dict(run.get("adaptive_results", {}))
        ar["qps"] = None
        rows.append(row(f"Adaptive target={target} avgK={ar.get('avg_k', float('nan')):.1f}", ds, ar, "cached", run_name))
print_table(rows, "08 Adaptive-K Cached Candidate Results")



08 Adaptive-K Cached Candidate Results
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
Fixed K=100                                10k     0.9963  0.9724  0.8641       -         -         -         -         - cached        10k_harddist_adaptive_20260427_134359
Fixed K=200                                10k     0.9966  0.9974  0.9862       -         -         -         -         - cached        10k_harddist_adaptive_20260427_134359
Fixed K=500                                10k     0.9966  0.9986  0.9989  0.9678         -         -         -         - cached        10k_harddist_adaptive_20260427_134359
Fixed K=1000                               10k     0.9966  0.9986  0.9989  0.9888         -         -         -         - cached        10k_harddi

## 08: Listwise WJ Distillation Ablation

Listwise distillation was tested as a KDIndex-style objective. It did not improve over hard-negative distillation on 10k.


In [10]:
rows = []
lw = results["listwise_wjdistill"]
if lw:
    for run_name, run in lw.get("runs", {}).items():
        ds = run.get("config", {}).get("dataset_name", "?")
        for group_name in ["start_eval", "listwise_eval"]:
            label = "Start" if group_name == "start_eval" else "Listwise"
            for name, res in run.get(group_name, {}).items():
                rows.append(row(label + " " + name, ds, res, "CPU+GPU", run_name))
print_table(rows, "09 Listwise WJ Distillation Results")



09 Listwise WJ Distillation Results
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
Start k100_wj_rerank                       10k     0.9962  0.9724  0.8641       -    9398.7       0.2         -      26.3 CPU+GPU       10k_listwise_20260427_134917
Start k200_wj_rerank                       10k     0.9966  0.9974  0.9862       -    7259.4       0.2         -      26.3 CPU+GPU       10k_listwise_20260427_134917
Start k500_wj_rerank                       10k     0.9966  0.9986  0.9989  0.9679    3811.1       0.2         -      26.3 CPU+GPU       10k_listwise_20260427_134917
Listwise k100_wj_rerank                    10k     0.9963  0.9674  0.8506       -   10689.6       0.2         -       3.1 CPU+GPU       10k_listwise_20260427_134917
Listwise k2

## 09: Raw Polygon PointNet Ablation

This experiment asked whether raw WKT coordinate sequences could replace quadtree vectors as the learned candidate-generator input. It was trained with WJ-aligned hard negatives and teacher embedding preservation, but recall remained very low. This supports keeping quadtree occupancy as an important inductive bias.


In [11]:
rows = []
raw = results["raw_pointnet"]
if raw:
    for run_name, run in raw.get("runs", {}).items():
        for name, res in run.get("raw_results", {}).items():
            rr = dict(res.get("wj_rerank", {}))
            rr.update({"qps": res.get("qps"), "build_s": res.get("build_s")})
            rows.append(row("Raw PointNet + WJ " + name, "10k", rr, "CPU+GPU", run_name))
print_table(rows, "10 Raw Geometry PointNet Ablation")



10 Raw Geometry PointNet Ablation
Method                                     Data      R@10    R@50   R@100   R@500       QPS     Build     VecMB     IdxMB HW            Note
------------------------------------------------------------------------------------------------------------------------------------------------------
Raw PointNet + WJ k100                     10k     0.0141  0.0137  0.0137       -    5908.8       0.3         -         - CPU+GPU       10k_raw_pointnet_20260427_164655
Raw PointNet + WJ k200                     10k     0.0316  0.0290  0.0288       -    4318.3       0.3         -         - CPU+GPU       10k_raw_pointnet_20260427_164655
Raw PointNet + WJ k500                     10k     0.0830  0.0732  0.0719  0.0711    3227.1       0.3         -         - CPU+GPU       10k_raw_pointnet_20260427_164655


## Final Takeaways

1. The original WJ baseline has the best deep `R@500`, but it is memory-heavy and slow to build.
2. MLP cosine alone is fast but not recall-competitive.
3. Exact original-WJ reranking is the critical step: it converts a learned candidate generator into a high-recall retrieval system.
4. Hard-negative WJ distillation is the strongest learning improvement because it trains on the candidate generator's actual false positives.
5. Adaptive-K gives a tunable compute/recall tradeoff by lowering average rerank budget.
6. Neural MinHash and listwise distillation are useful ablations but do not beat HardDist MLP.
7. Raw PointNet from WKT coordinates performs poorly, suggesting that quadtree occupancy remains a necessary inductive bias for this WJ-aligned GT.

Recommended final method to report:

- **10k**: HardDist MLP + GPU exact WJ rerank, `K=500`.
- **Full**: HardDist MLP + GPU exact WJ rerank, `K=1000` for speed/top-100 or `K=2000` for better deep recall.
- **Optional systems extension**: Adaptive-K with target `R@100=0.998` on 10k.
